# 00 - Variables del Workshop BNCR

> **Importante:** Este notebook solo define variables. No reinicia el kernel.

Ejecute **todas las celdas** antes de continuar con `00_setup`.


In [ ]:
catalog_name = "BNS"
schema_raw = "raw"
schema_bronze = "bronze"
schema_silver = "silver"
schema_gold = "gold"
volume = "transacciones"

vol_path = f"/Volumes/{catalog_name}/{schema_raw}/{volume}"
repo_url = "https://github.com/Ricojacob01/Latam_resources_spanish"
repo_path = "Data Engineering"

print(f"Catálogo   : {catalog_name}")
print(f"Volumen    : {vol_path}")
print(f"Repositorio: {repo_url}/{repo_path}")


In [ ]:
def carga_datos(carga):
    """Carga archivos initial o incremental desde GitHub al Volume UC."""
    import git, os, tempfile

    if carga not in ("initial", "incremental"):
        raise ValueError("carga debe ser 'initial' o 'incremental'")

    dbutils.fs.mkdirs(vol_path)

    with tempfile.TemporaryDirectory() as tmp_dir:
        repo = git.Repo.clone_from(repo_url, tmp_dir, depth=1, no_checkout=True)
        repo.git.checkout("HEAD", "--", f"{repo_path}/Files/{carga}")
        src = os.path.join(tmp_dir, repo_path, "Files", carga)

        for item in os.listdir(src):
            local_path = os.path.join(src, item)
            dst = f"{vol_path}/{item}"
            try:
                dbutils.fs.rm(dst, recurse=True)
            except Exception:
                pass
            dbutils.fs.cp(f"file:{local_path}", dst, recurse=os.path.isdir(local_path))

    print(f"Carga '{carga}' completada en {vol_path}")
    display(dbutils.fs.ls(vol_path))
